# Chapter 09 (Hands-on, LangChain) - Calculator, Search & Database Tools + Handling Tool Errors

This is the **LangChain** version of the Chapter 09 notebook. Same tools, same failures, same lessons - only the plumbing changes.

**What you will build**
1. Three real tools with the `@tool` decorator: a **calculator**, a **search** tool (Wikipedia) and a **database** tool (SQLite)
2. A **safe tool executor** that always returns a `ToolMessage` and never crashes the agent loop
3. An agent loop that lets the model **see tool errors and correct itself** - first written by hand, then with LangChain's prebuilt `create_agent` + middleware

**What LangChain changes compared with the raw OpenAI SDK**

| Raw OpenAI SDK | LangChain |
|---|---|
| Hand-written JSON schema per tool | `@tool` builds the schema from type hints + docstring |
| `client.chat.completions.create(tools=...)` | `llm.bind_tools(TOOLS)` |
| `json.loads(call.function.arguments)` | `message.tool_calls` is already parsed; broken JSON lands in `message.invalid_tool_calls` |
| `{"role": "tool", ...}` dict | `ToolMessage(content=..., tool_call_id=...)` |
| Hand-written loop | Hand-written loop **or** `create_agent(...)` |

**Running in VS Code**
1. Install the *Python* and *Jupyter* extensions.
2. Open this file, click **Select Kernel** (top right) and pick your Python environment / venv.
3. Create a `.env` file next to this notebook containing: `OPENAI_API_KEY=sk-...`
4. Run the cells top to bottom.

## 1. Setup

In [ ]:
# %pip installs into the kernel selected in VS Code
# langchain >= 1.0 is needed for create_agent and middleware (section 9)
%pip install -q -U langchain langchain-openai python-dotenv requests

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # reads .env from the notebook's folder

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
import json
from langchain_openai import ChatOpenAI

MODEL = "gpt-4.1-mini"   # any tool-calling chat model works
llm = ChatOpenAI(model=MODEL, max_tokens=1000)   # picks up OPENAI_API_KEY from the environment

## 2. Tool 1 - Calculator

LLMs are unreliable at arithmetic, so we hand maths to Python.

We do **not** use `eval()` - it would run any code the model writes. Instead we parse the expression and allow only numbers, arithmetic operators and a few maths functions.

`@tool` turns a normal Python function into a LangChain tool:
- the **function name** becomes the tool name
- the **docstring** becomes the description the model reads
- the **type hints** become the argument schema

In [ ]:
import ast
import math
import operator
from langchain_core.tools import tool

_OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}
_FUNCTIONS = {"sqrt": math.sqrt, "abs": abs, "round": round, "min": min, "max": max}


def _evaluate(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPERATORS:
        left, right = _evaluate(node.left), _evaluate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("Exponent too large (max 100)")
        return _OPERATORS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPERATORS:
        return _OPERATORS[type(node.op)](_evaluate(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _FUNCTIONS:
        return _FUNCTIONS[node.func.id](*[_evaluate(a) for a in node.args])
    raise ValueError("Only numbers, + - * / // % ** and sqrt/abs/round/min/max are allowed")


@tool
def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression such as '(1200 * 18) / 100'. Use this for ALL maths.
    Supports + - * / // % ** and sqrt, abs, round, min, max."""
    tree = ast.parse(expression, mode="eval")
    return _evaluate(tree.body)

In [ ]:
# Try it directly (no LLM involved).
# A LangChain tool is called with .invoke() and a dict of arguments.
print(calculator.invoke({"expression": "(1200 * 18) / 100"}))
print(calculator.invoke({"expression": "sqrt(144) + 2 ** 5"}))

## 3. Tool 2 - Search

We use the public **Wikipedia search API**: free, no API key, no sign-up.

Network tools fail in ways local tools do not (timeouts, rate limits, server errors), so the function sets a `timeout` and calls `raise_for_status()`.

In [ ]:
import re
import requests


@tool
def search(query: str, max_results: int = 3) -> list:
    """Search Wikipedia for general knowledge. Returns a list of {title, snippet} results.

    Args:
        query: Search keywords
        max_results: Number of results (default 3)
    """
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": max_results,
            "format": "json",
        },
        headers={"User-Agent": "genai-chap09-training-notebook/1.0"},
        timeout=10,
    )
    response.raise_for_status()  # turns HTTP 4xx/5xx into an exception

    hits = response.json()["query"]["search"]
    return [
        {"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])}
        for h in hits
    ]

In [ ]:
# Try it directly (needs internet)
search.invoke({"query": "ReAct agent large language model"})

## 4. Tool 3 - Database

A small in-memory **SQLite** database with two tables. The model writes SQL, we run it.

Guard rails:
- only a single `SELECT` statement is allowed (the model must never be able to `DROP` or `UPDATE`)
- results are capped at 20 rows so a careless `SELECT *` cannot flood the context

The **schema goes in the docstring** - without it the model has to guess table and column names.

In [ ]:
import sqlite3

# check_same_thread=False: the prebuilt agent in section 9 may run tools in a worker thread
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name       TEXT,
    category   TEXT,
    price      REAL,
    stock      INTEGER
);
CREATE TABLE orders (
    order_id   INTEGER PRIMARY KEY,
    product_id INTEGER REFERENCES products(product_id),
    customer   TEXT,
    city       TEXT,
    quantity   INTEGER,
    order_date TEXT
);
INSERT INTO products VALUES
    (1, 'Laptop',     'Electronics', 65000, 12),
    (2, 'Headphones', 'Electronics',  2500, 40),
    (3, 'Office Chair','Furniture',   8500,  7),
    (4, 'Desk Lamp',  'Furniture',    1200,  0),
    (5, 'Notebook',   'Stationery',     80, 300);
INSERT INTO orders VALUES
    (101, 1, 'Asha',   'Bengaluru', 1, '2026-10-02'),
    (102, 2, 'Rahul',  'Mumbai',    2, '2026-10-05'),
    (103, 5, 'Asha',   'Bengaluru', 10,'2026-10-09'),
    (104, 3, 'Meera',  'Delhi',     1, '2026-10-15'),
    (105, 2, 'Imran',  'Bengaluru', 1, '2026-11-01'),
    (106, 1, 'Meera',  'Delhi',     2, '2026-11-03');
""")

MAX_ROWS = 20


@tool
def query_database(sql: str) -> list:
    """Run one read-only SQLite SELECT query on the shop database and return the rows.
    Tables: products(product_id, name, category, price, stock),
    orders(order_id, product_id, customer, city, quantity, order_date 'YYYY-MM-DD').
    Prices are in INR.

    Args:
        sql: A single SELECT statement
    """
    statement = sql.strip().rstrip(";")
    if not statement.lower().startswith("select") or ";" in statement:
        raise PermissionError("Only a single SELECT statement is allowed")

    rows = conn.execute(statement).fetchmany(MAX_ROWS)
    return [dict(row) for row in rows]

In [ ]:
# Try it directly
query_database.invoke({"sql": "SELECT name, price, stock FROM products WHERE category = 'Electronics'"})

## 5. Describe the tools to the model

With the raw SDK we wrote a JSON schema for every tool by hand. LangChain generates it from the function.

The description is still the only thing the model sees, so the docstring matters just as much. Print the generated schema and check it says what you intended.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

TOOLS = [calculator, search, query_database]

# Maps tool names (strings) to the LangChain tool objects
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

# This is exactly what gets sent to the model
for t in TOOLS:
    print(json.dumps(convert_to_openai_tool(t), indent=2))
    print("-" * 60)

In [ ]:
# bind_tools attaches the tool schemas to every call made through this model object
llm_with_tools = llm.bind_tools(TOOLS)

# One call, no loop: the model answers with tool_calls instead of text
reply = llm_with_tools.invoke("What is 18% of 1250?")
print("content    :", repr(reply.content))
print("tool_calls :", reply.tool_calls)   # already parsed: [{'name': ..., 'args': {...}, 'id': ...}]

## 6. What can go wrong with a tool call?

| # | Problem | Example | What LangChain does by default |
|---|---------|---------|-------------------------------|
| 1 | Model calls a tool that does not exist | `get_stock_price` | Nothing - your lookup fails with `KeyError` |
| 2 | Arguments are not valid JSON | `{"expression": "2+2"` | Puts the call in `message.invalid_tool_calls` (easy to forget!) |
| 3 | Wrong or missing arguments | `calculator(expr="2+2")` | `tool.invoke` raises a pydantic `ValidationError` |
| 4 | The tool itself raises an error | division by zero, bad SQL, API timeout | The exception propagates |
| 5 | The tool "succeeds" but the result is unexpected | empty result, huge result | Nothing - it is passed on as-is |

In a basic loop, any of 1-4 raises a Python exception and **the whole agent crashes**. See it happen:

In [ ]:
# The naive way - this is what a basic tool-calling loop does
naive_calls = [
    {"name": "calculator",      "args": {"expression": "100 / 0"}},   # tool raises
    {"name": "calculator",      "args": {"expr": "2+2"}},             # wrong argument name
    {"name": "get_stock_price", "args": {"ticker": "TCS"}},           # unknown tool
]

for call in naive_calls:
    try:
        TOOLS_BY_NAME[call["name"]].invoke(call["args"])
    except Exception as e:
        print(f"Agent crashed with: {type(e).__name__} - {str(e)[:120]}")

## 7. A safe tool executor

Rule: **a tool call must always produce a `ToolMessage`**, even when it fails.

- Success -> `{"ok": true, "result": ...}` with `status="success"`
- Failure -> `{"ok": false, "error": "..."}` with `status="error"` and a message the model can act on

The error goes back to the model as the *observation*. The model can then fix its arguments and try again, or explain the problem to the user.

`args` may be a **dict** (a normal `tool_calls` entry) or a **raw string** (an `invalid_tool_calls` entry, or a test).

In [ ]:
from langchain_core.messages import ToolMessage
from pydantic import ValidationError

MAX_RESULT_CHARS = 2000   # keep huge tool outputs from flooding the context
MAX_RETRIES = 2           # retries for temporary network problems


def _tool_message(call_id: str, name: str, payload: dict) -> ToolMessage:
    return ToolMessage(
        content=json.dumps(payload, default=str),
        tool_call_id=call_id,
        name=name,
        status="success" if payload["ok"] else "error",
    )


def execute_tool(name: str, args, call_id: str = "test") -> ToolMessage:
    """Run a tool safely. ALWAYS returns a ToolMessage - never raises."""

    # Problem 1: unknown tool
    if name not in TOOLS_BY_NAME:
        return _tool_message(call_id, name, {
            "ok": False,
            "error": f"Unknown tool '{name}'. Available tools: {list(TOOLS_BY_NAME)}",
        })

    # Problem 2: arguments are not valid JSON (raw string from invalid_tool_calls)
    if not isinstance(args, dict):
        try:
            args = json.loads(args or "{}")
            if not isinstance(args, dict):
                raise ValueError("arguments must be a JSON object")
        except ValueError as e:   # json.JSONDecodeError is a ValueError
            return _tool_message(call_id, name, {"ok": False, "error": f"Invalid JSON arguments: {e}"})

    # Problems 3 and 4: wrong arguments, or the tool raises
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            result = TOOLS_BY_NAME[name].invoke(args)
            break
        except ValidationError as e:
            # wrong / missing argument names or types - LangChain validates against the schema
            problems = "; ".join(
                f"{'.'.join(str(p) for p in err['loc'])}: {err['msg']}" for err in e.errors()
            )
            expected = list(TOOLS_BY_NAME[name].args)
            return _tool_message(call_id, name, {
                "ok": False,
                "error": f"Bad arguments for {name}: {problems}. Expected arguments: {expected}",
            })
        except (requests.Timeout, requests.ConnectionError) as e:
            # temporary problem -> retry, then give up politely
            if attempt > MAX_RETRIES:
                return _tool_message(call_id, name, {
                    "ok": False,
                    "error": f"{name} is unavailable right now ({type(e).__name__}). Do not retry.",
                })
        except Exception as e:
            # everything else: ZeroDivisionError, sqlite3.Error, ValueError, HTTP errors ...
            return _tool_message(call_id, name, {"ok": False, "error": f"{type(e).__name__}: {e}"})

    # Problem 5: unexpected results
    if result is None or result == [] or result == "":
        return _tool_message(call_id, name, {
            "ok": True,
            "result": [],
            "note": "The tool ran successfully but returned no data. Do not invent an answer.",
        })

    payload = json.dumps({"ok": True, "result": result}, default=str)
    if len(payload) > MAX_RESULT_CHARS:
        return _tool_message(call_id, name, {
            "ok": True,
            "result": payload[:MAX_RESULT_CHARS],
            "note": "Result was truncated. Use a more specific query.",
        })
    return _tool_message(call_id, name, {"ok": True, "result": result})

### Test every failure without spending a single token

`execute_tool` takes a name and arguments, so we can test it without calling the LLM.

In [ ]:
tests = [
    ("Happy path",            "calculator",      {"expression": "(1200 * 18) / 100"}),
    ("1. Unknown tool",       "get_stock_price", {"ticker": "TCS"}),
    ("2. Invalid JSON",       "calculator",      '{"expression": "2+2"'),          # raw string, missing }
    ("3. Wrong argument name","calculator",      {"expr": "2+2"}),
    ("3. Missing argument",   "query_database",  {}),
    ("4. Tool raises (maths)","calculator",      {"expression": "100 / 0"}),
    ("4. Tool raises (SQL)",  "query_database",  {"sql": "SELECT phone FROM orders"}),
    ("4. Blocked query",      "query_database",  {"sql": "DROP TABLE orders"}),
    ("4. Unsafe expression",  "calculator",      {"expression": "__import__('os').getcwd()"}),
    ("5. Empty result",       "query_database",  {"sql": "SELECT * FROM orders WHERE city = 'Chennai'"}),
]

for label, name, args in tests:
    msg = execute_tool(name, args)
    print(f"{label:26} [{msg.status:7}] -> {msg.content}")

### Simulate a flaky API

A fake tool that times out on the first two calls and then works. `execute_tool` retries it automatically.

In [ ]:
_calls = {"count": 0}

@tool
def flaky_search(query: str) -> list:
    """A search tool that fails twice before it works."""
    _calls["count"] += 1
    if _calls["count"] <= 2:
        raise requests.Timeout("simulated timeout")
    return [{"title": "It worked", "snippet": f"Succeeded on attempt {_calls['count']}"}]

TOOLS_BY_NAME["flaky_search"] = flaky_search          # register temporarily
print(execute_tool("flaky_search", {"query": "test"}).content)
del TOOLS_BY_NAME["flaky_search"]                     # clean up

### LangChain's built-in switches

LangChain tools also have two per-tool options. They are handy for simple cases:

- `handle_tool_error` - catches a `ToolException` raised inside the tool and returns its text instead
- `handle_validation_error` - returns a message instead of raising when the arguments do not match the schema

Limits to know: `handle_tool_error` catches **only** `ToolException` (a `ZeroDivisionError` still crashes), and neither option covers unknown tools, invalid JSON, retries or empty results. That is why we keep our own `execute_tool`.

In [ ]:
from langchain_core.tools import StructuredTool, ToolException


def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ToolException("Cannot divide by zero. Ask the user for a non-zero divisor.")
    return a / b


divide_tool = StructuredTool.from_function(
    func=divide,
    handle_tool_error=True,                                            # ToolException -> returned as text
    handle_validation_error="Bad arguments. Expected numbers 'a' and 'b'.",
)

print(divide_tool.invoke({"a": 10, "b": 4}))
print(divide_tool.invoke({"a": 10, "b": 0}))      # error text, no crash
print(divide_tool.invoke({"a": "ten"}))           # validation message, no crash

## 8. The agent loop with error handling

The same ReAct loop as before, written with LangChain messages.

Changes compared with the basic loop:
- tools run through `execute_tool`, so the loop cannot crash on a tool failure
- **`invalid_tool_calls` are answered too** - the API rejects the next request if any tool call is left without a tool message
- a **system prompt** tells the model what to do when a tool reports an error
- a **consecutive error limit** stops the model from retrying a broken call forever
- the `for ... else` reports when `MAX_STEPS` is reached without an answer

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
- Use the calculator for all arithmetic.
- Every tool result is JSON with an "ok" field.
- If "ok" is false, read the error, correct your tool call and try again.
- If the same problem repeats, or a tool returns no data, tell the user honestly.
- Never invent tool results."""

MAX_STEPS = 8
MAX_CONSECUTIVE_ERRORS = 3


def run_agent(question: str) -> str:
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]
    consecutive_errors = 0

    for step in range(MAX_STEPS):
        print(f"==== Step {step} ====")

        message = llm_with_tools.invoke(messages)   # returns an AIMessage
        messages.append(message)

        # tool_calls have parsed dict args; invalid_tool_calls have the raw (broken) string
        calls = list(message.tool_calls) + list(message.invalid_tool_calls)

        # No tool calls -> final answer
        if not calls:
            print(f"Final Output: {message.content}")
            return message.content

        # Every tool call MUST get a ToolMessage back, success or failure
        for call in calls:
            observation = execute_tool(call["name"], call["args"], call["id"])
            messages.append(observation)
            print(f"Tool call -> {call['name']} {call['args']}")
            print(f"Observation -> {observation.content}")

            if observation.status == "success":
                consecutive_errors = 0
            else:
                consecutive_errors += 1

        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            print("Stopped: too many tool errors in a row")
            return "Sorry, I could not complete this because a tool kept failing."
    else:
        print("Stopped: reached MAX_STEPS without a final answer")
        return "Sorry, I could not finish within the step limit."

## 9. The same agent with `create_agent` + middleware

LangChain ships a prebuilt agent loop: `create_agent`. You give it a model, tools and a system prompt, and it runs the model -> tools -> model cycle for you.

By default an exception raised inside a tool body (division by zero, bad SQL, a timeout) propagates and stops the run. **Middleware** lets us wrap every tool call - the same idea as `execute_tool`, plugged into the prebuilt loop:

- `@wrap_tool_call` receives the `request` (which tool, which args) and a `handler` that actually runs the tool
- we call `handler(request)` inside `try/except` and turn any exception into an error `ToolMessage`

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def safe_tool_calls(request, handler):
    """Wrap every tool call: retry temporary failures, turn exceptions into error messages."""
    name = request.tool_call["name"]
    call_id = request.tool_call["id"]

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            result = handler(request)          # runs the tool, returns a ToolMessage
            break
        except (requests.Timeout, requests.ConnectionError) as e:
            if attempt > MAX_RETRIES:
                return _tool_message(call_id, name, {
                    "ok": False,
                    "error": f"{name} is unavailable right now ({type(e).__name__}). Do not retry.",
                })
        except Exception as e:
            return _tool_message(call_id, name, {"ok": False, "error": f"{type(e).__name__}: {e}"})

    # Problem 5: worked but empty
    if isinstance(result, ToolMessage) and result.content in ("", "[]", "null", "None"):
        return _tool_message(call_id, name, {
            "ok": True,
            "result": [],
            "note": "The tool ran successfully but returned no data. Do not invent an answer.",
        })
    return result


# The prebuilt agent passes raw tool output to the model, so we describe errors a little differently
AGENT_PROMPT = """You are a helpful assistant with access to tools.
- Use the calculator for all arithmetic.
- If a tool result is JSON with "ok": false, read the error, correct your tool call and try again.
- If the same problem repeats, or a tool returns no data, tell the user honestly.
- Never invent tool results."""

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=AGENT_PROMPT,
    middleware=[safe_tool_calls],
)


def run_prebuilt_agent(question: str) -> str:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        {"recursion_limit": 2 * MAX_STEPS + 1},   # step cap: each model call + tool run is 2 graph steps
    )
    for m in result["messages"]:
        m.pretty_print()                           # full trace: human, AI tool calls, tool results, answer
    return result["messages"][-1].content

Which one should you use?

- **Hand-written loop (section 8)** - you see and control every step. Best for learning and for unusual control flow.
- **`create_agent` (section 9)** - less code, plus streaming, memory and ready-made middleware (for example `ToolRetryMiddleware` and `ToolCallLimitMiddleware` in `langchain.agents.middleware`). Best for real projects.

The cells below use `run_agent`. Swap in `run_prebuilt_agent` to compare the traces.

## 10. Try it - normal cases

In [ ]:
# Calculator
run_agent("What is 18% GST on 1,250 rupees, and what is the total?")

In [ ]:
# Search
run_agent("Who created the Python programming language and in which year was it first released?")

In [ ]:
# Database
run_agent("Which products are out of stock?")

In [ ]:
# All three tools together
run_agent("What is the total revenue from orders placed in Bengaluru? Then add 18% GST to it.")

In [ ]:
# Same question through the prebuilt agent
run_prebuilt_agent("What is the total revenue from orders placed in Bengaluru? Then add 18% GST to it.")

## 11. Try it - error cases

Watch the `Observation` lines. The tool reports the problem and the model decides what to do next.

In [ ]:
# Tool raises an error -> model should explain instead of crashing
run_agent("Use the calculator to divide 100 by 0.")

In [ ]:
# Column does not exist -> the SQL error should make the model give an honest answer
run_agent("What is the phone number of the customer Asha?")

In [ ]:
# Empty result -> model should say there is no data, not make something up
run_agent("List all orders from Chennai.")

In [ ]:
# Blocked operation -> the guard rail rejects it
run_agent("Delete all orders from the database.")

In [ ]:
# Error case through the prebuilt agent - the middleware catches the exception
run_prebuilt_agent("Use the calculator to divide 100 by 0.")

## 12. Exercises

1. Add a fourth tool, `get_weather(city)`, with `@tool`, using the Open-Meteo API from the previous notebook. What should it return for a city that does not exist? (Remember to add it to `TOOLS`, rebuild `TOOLS_BY_NAME` and call `llm.bind_tools` again.)
2. Lower `MAX_CONSECUTIVE_ERRORS` to 1 and re-run the error cases. What changes?
3. Remove the schema from the `query_database` docstring and ask the Bengaluru revenue question again. How many extra steps does the model need?
4. Add a `category` filter that is spelled wrongly in the question ("Electronic items") and see whether the model recovers from the empty result.
5. Remove `middleware=[safe_tool_calls]` from `create_agent` and re-run the divide-by-zero case. What happens, and why?
6. Replace the retry code in `safe_tool_calls` with LangChain's built-in `ToolRetryMiddleware`. Which failures should it retry, and which should it not?

## Key takeaways
- A tool call must **always** return a `ToolMessage` - catch exceptions and send the error text back.
- `@tool` writes the schema for you, but the **docstring is still the prompt** - print the schema and read it.
- Do not forget `invalid_tool_calls`: broken JSON never reaches your tool, but it still needs an answer.
- Write error messages **for the model**: say what was wrong and what is allowed.
- Treat "worked but empty / too large" as a separate case from "failed".
- Retry only temporary failures, and always cap retries and steps.
- Guard rails (safe maths parser, SELECT-only SQL) belong in the tool, not in the prompt.
- The framework changes the plumbing, not the principles: the hand-written loop and `create_agent` + middleware apply the same rules.